In [1]:
import re
import numpy as np
import pandas as pd

In [24]:
import os
import re
import pandas as pd

# File path
file_path = r"D:\projects\library search engine\backend\process_raw_data\final_book_dataset_kaggle2.csv"
output_path = r"D:\projects\library search engine\backend\process_raw_data\data1.csv"

# 1. Load raw dataset as strings to prevent float/scientific notation conversions
df = pd.read_csv(file_path, dtype=str)

cleaned_df = pd.DataFrame()


# Helper: Clean Author strings
def clean_author(val):
    if pd.isna(val) or not str(val).strip():
        return ""
    val_str = str(val).strip()
    val_str = re.sub(r"^[\[\s\"']+|[\]\s\"']+$", "", val_str)
    val_str = re.sub(r"\s+and\s+,", ",", val_str)
    return re.sub(r"\s+", " ", val_str).strip()


# Helper: Extract 4-digit Published Year
def extract_year(val):
    if pd.isna(val):
        return ""
    match = re.search(r"\b(18|19|20)\d{2}\b", str(val))
    return match.group(0) if match else ""


# Helper: Clean and sanitize ISBN
def clean_isbn(val):
    if pd.isna(val):
        return ""
    val_str = str(val).strip().upper()

    if val_str.lower() in ["nan", "none", "<na>", "", "0"]:
        return ""

    # Handle float or scientific notation strings if present
    if "." in val_str or "E" in val_str:
        try:
            val_str = str(int(float(val_str)))
        except (ValueError, OverflowError):
            if val_str.endswith(".0"):
                val_str = val_str[:-2]

    cleaned = re.sub(r"[^\dX]", "", val_str)
    return cleaned if len(cleaned) >= 9 else ""


# --- FIELD MAPPINGS ---

cleaned_df["title"] = df.get("title", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
cleaned_df["author"] = df.get("author", pd.Series([""] * len(df))).apply(clean_author)
cleaned_df["categories"] = ""
cleaned_df["thumbnail"] = ""
cleaned_df["description"] = df.get("description", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()

# Page normalization: Clean non-negative integer string or empty string ""
if "pages" in df.columns:
    pages_num = pd.to_numeric(df["pages"], errors="coerce")
    cleaned_df["pages"] = pages_num.apply(
        lambda v: str(int(v)) if pd.notna(v) and v >= 0 else ""
    )
else:
    cleaned_df["pages"] = ""

cleaned_df["publisher"] = df.get("publisher", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
cleaned_df["language"] = df.get("language", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()

# Safely select link fallback
complete_link = df.get("complete_link", pd.Series([None] * len(df)))
link = df.get("link", pd.Series([None] * len(df)))
cleaned_df["link"] = complete_link.fillna(link).fillna("").astype(str).str.strip()

# Published Year extracted from publisher field
cleaned_df["published_year"] = cleaned_df["publisher"].apply(extract_year)

# Clean ISBN (Check ISBN_13 first, then ISBN fallback if available)
isbn13_series = df.get("ISBN_13", df.get("isbn13", pd.Series([None] * len(df))))
isbn_series = df.get("ISBN", df.get("isbn", pd.Series([None] * len(df))))

cleaned_df["isbn"] = [
    clean_isbn(isbn13) or clean_isbn(isbn)
    for isbn13, isbn in zip(isbn13_series, isbn_series)
]

# Ensure exact Zod schema column ordering
schema_fields = [
    "title",
    "author",
    "categories",
    "thumbnail",
    "description",
    "pages",
    "publisher",
    "language",
    "link",
    "published_year",
    "isbn",
]
cleaned_df = cleaned_df[schema_fields]


# --- STRICT ZOD SCHEMA VALIDATION FILTER ---
valid_title_mask = cleaned_df["title"].str.len() >= 3
valid_author_mask = cleaned_df["author"].str.len() >= 3
valid_isbn_mask = cleaned_df["isbn"].str.len() >= 1

valid_books_df = cleaned_df[
    valid_title_mask & valid_author_mask & valid_isbn_mask
].reset_index(drop=True)

# Export processed CSV
os.makedirs(os.path.dirname(output_path), exist_ok=True)
valid_books_df.to_csv(output_path, index=False)

print(f"Total raw rows: {len(df)}")
print(f"Valid schema-compliant rows saved to data1.csv: {len(valid_books_df)}")

Total raw rows: 830
Valid schema-compliant rows saved to data1.csv: 394


In [23]:
import os
import re
import pandas as pd

# File path
file_path = r"D:\projects\library search engine\backend\process_raw_data\GoodReads_100k_books.csv"
output_path = r"D:\projects\library search engine\backend\process_raw_data\data2.csv"

# 1. Load raw dataset as strings to prevent scientific notation (e.g., 9.78E+12)
df = pd.read_csv(file_path, dtype=str)

cleaned_df = pd.DataFrame()


# Helper: Clean Author strings
def clean_author(val):
    if pd.isna(val) or not str(val).strip():
        return ""
    val_str = str(val).strip()
    val_str = re.sub(r"^[\[\s\"']+|[\]\s\"']+$", "", val_str)
    return re.sub(r"\s+", " ", val_str).strip()


# Helper: Extract 4-digit Year from description/text
def extract_year(val):
    if pd.isna(val):
        return ""
    match = re.search(r"\b(18|19|20)\d{2}\b", str(val))
    return match.group(0) if match else ""


# Helper: Clean ISBN (handles floats, scientific notation strings, and nulls)
def clean_isbn(raw_isbn, raw_isbn13):
    def sanitize(val):
        if pd.isna(val):
            return ""
        val_str = str(val).strip().upper()

        if val_str.lower() in ["nan", "none", "<na>", "", "0"]:
            return ""

        # Handle float or scientific notation strings if present (e.g. "9.78014E+12")
        if "." in val_str or "E" in val_str:
            try:
                val_str = str(int(float(val_str)))
            except (ValueError, OverflowError):
                if val_str.endswith(".0"):
                    val_str = val_str[:-2]

        cleaned = re.sub(r"[^\dX]", "", val_str)
        return cleaned

    isbn13_clean = sanitize(raw_isbn13)
    if len(isbn13_clean) >= 9:
        return isbn13_clean

    isbn_clean = sanitize(raw_isbn)
    if len(isbn_clean) >= 9:
        return isbn_clean

    return ""


# --- FIELD MAPPINGS ---

cleaned_df["title"] = df["title"].fillna("").astype(str).str.strip()
cleaned_df["author"] = df["author"].apply(clean_author)
cleaned_df["categories"] = df.get("genre", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
cleaned_df["thumbnail"] = df.get("img", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
cleaned_df["description"] = df.get("desc", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()

# Pages: Non-negative integer string or empty string ""
if "pages" in df.columns:
    pages_num = pd.to_numeric(df["pages"], errors="coerce")
    cleaned_df["pages"] = pages_num.apply(
        lambda v: str(int(v)) if pd.notna(v) and v >= 0 else ""
    )
else:
    cleaned_df["pages"] = ""

cleaned_df["publisher"] = ""
cleaned_df["language"] = ""
cleaned_df["link"] = df.get("link", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()

# Extracted Year from description text if available
desc_col = df.get("desc", pd.Series([""] * len(df)))
cleaned_df["published_year"] = desc_col.apply(extract_year)

# Clean Required ISBN
isbn_col = df.get("isbn", pd.Series([None] * len(df)))
isbn13_col = df.get("isbn13", pd.Series([None] * len(df)))
cleaned_df["isbn"] = [
    clean_isbn(isbn, isbn13) for isbn, isbn13 in zip(isbn_col, isbn13_col)
]

# Ensure exact Zod schema column ordering
schema_fields = [
    "title",
    "author",
    "categories",
    "thumbnail",
    "description",
    "pages",
    "publisher",
    "language",
    "link",
    "published_year",
    "isbn",
]
cleaned_df = cleaned_df[schema_fields]


# --- STRICT ZOD SCHEMA VALIDATION FILTER ---
valid_title_mask = cleaned_df["title"].str.len() >= 3
valid_author_mask = cleaned_df["author"].str.len() >= 3
valid_isbn_mask = cleaned_df["isbn"].str.len() >= 1

valid_books_df = cleaned_df[
    valid_title_mask & valid_author_mask & valid_isbn_mask
].reset_index(drop=True)

# Save processed CSV
os.makedirs(os.path.dirname(output_path), exist_ok=True)
valid_books_df.to_csv(output_path, index=False)

print(f"Total raw rows: {len(df)}")
print(f"Valid schema-compliant rows saved to data2.csv: {len(valid_books_df)}")

Total raw rows: 100000
Valid schema-compliant rows saved to data2.csv: 89887


In [21]:
import os
import re
import pandas as pd

# File paths
input_file_path = (
    r"D:\projects\library search engine\backend\process_raw_data\Goodreadss Books.csv"
)
output_file_path = r"D:\projects\library search engine\backend\process_raw_data\data3.csv"

# 1. Load raw dataset as strings to prevent scientific notation (e.g. 9.78E+12)
df = pd.read_csv(input_file_path, dtype=str)

cleaned_df = pd.DataFrame()


# Helper: Clean Author strings
def clean_author(val):
    if pd.isna(val) or not str(val).strip():
        return ""
    val_str = str(val).strip()
    val_str = re.sub(r"^[\[\s\"']+|[\]\s\"']+$", "", val_str)
    return re.sub(r"\s+", " ", val_str).strip()


# Helper: Extract 4-digit Year
def extract_year(first_pub, pub):
    for val in [first_pub, pub]:
        if pd.notna(val) and str(val).strip():
            match = re.search(r"\b\d{4}\b", str(val))
            if match:
                return match.group(0)
    return ""


# Helper: Clean ISBN (handles floats, scientific notation strings, and nulls)
def clean_isbn(raw_isbn, raw_isbn13):
    def sanitize(val):
        if pd.isna(val):
            return ""
        val_str = str(val).strip().upper()

        if val_str.lower() in ["nan", "none", "<na>", "", "0"]:
            return ""

        # Handle float or scientific notation strings if present
        if "." in val_str or "E" in val_str:
            try:
                val_str = str(int(float(val_str)))
            except (ValueError, OverflowError):
                if val_str.endswith(".0"):
                    val_str = val_str[:-2]

        cleaned = re.sub(r"[^\dX]", "", val_str)
        return cleaned

    isbn13_clean = sanitize(raw_isbn13)
    if len(isbn13_clean) >= 9:
        return isbn13_clean

    isbn_clean = sanitize(raw_isbn)
    if len(isbn_clean) >= 9:
        return isbn_clean

    return ""


# --- FIELD MAPPINGS ---

cleaned_df["title"] = df["title"].fillna("").astype(str).str.strip()
cleaned_df["author"] = df["author"].apply(clean_author)
cleaned_df["categories"] = df.get("genres", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
cleaned_df["thumbnail"] = ""
cleaned_df["description"] = df.get("description", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()

# Page normalization -> Integer string or empty string ""
if "num_pages" in df.columns:
    pages_num = pd.to_numeric(df["num_pages"], errors="coerce")
    cleaned_df["pages"] = pages_num.apply(
        lambda v: str(int(v)) if pd.notna(v) and v >= 0 else ""
    )
else:
    cleaned_df["pages"] = ""

cleaned_df["publisher"] = ""
cleaned_df["language"] = df.get("language", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
cleaned_df["link"] = ""

# Extracted Published Year
first_pub_col = df.get("first_publish_date", pd.Series([None] * len(df)))
pub_col = df.get("publish_date", pd.Series([None] * len(df)))
cleaned_df["published_year"] = [
    extract_year(fp, p) for fp, p in zip(first_pub_col, pub_col)
]

# ISBN Cleaning
isbn_col = df.get("isbn", pd.Series([None] * len(df)))
isbn13_col = df.get("isbn13", pd.Series([None] * len(df)))
cleaned_df["isbn"] = [
    clean_isbn(isbn, isbn13) for isbn, isbn13 in zip(isbn_col, isbn13_col)
]


# --- STRICT ZOD SCHEMA VALIDATION FILTER ---
valid_title_mask = cleaned_df["title"].str.len() >= 3
valid_author_mask = cleaned_df["author"].str.len() >= 3
valid_isbn_mask = cleaned_df["isbn"].str.len() >= 1

valid_books_df = cleaned_df[
    valid_title_mask & valid_author_mask & valid_isbn_mask
].reset_index(drop=True)

# Save processed CSV
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
valid_books_df.to_csv(output_file_path, index=False)

print(f"Total raw records: {len(df)}")
print(f"Schema-compliant records saved to data3.csv: {len(valid_books_df)}")

Total raw records: 20068
Schema-compliant records saved to data3.csv: 19251


In [20]:
import os
import pandas as pd
import numpy as np

# File paths
input_path = r"D:\projects\library search engine\backend\process_raw_data\google_books_1299.csv"
output_path = r"D:\projects\library search engine\backend\process_raw_data\data4.csv"

# 1. Read Raw CSV as strings
df = pd.read_csv(input_path, dtype=str)

# 2. Map original headers to schema keys
col_map = {
    'title': 'title',
    'author': 'author',
    'generes': 'categories',
    'description': 'description',
    'publisher': 'publisher',
    'page_count': 'pages',
    'language': 'language',
    'ISBN': 'isbn'
}
df = df.rename(columns=col_map)

# 3. Clean String Fields & Strip Whitespace
str_cols = ['title', 'author', 'categories', 'description', 'publisher', 'language']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str).str.strip()
        df[col] = df[col].replace({'none': '', 'nan': '', 'None': '', '<NA>': ''})
    else:
        df[col] = ''

# 4. Fill missing optional string fields
df['thumbnail'] = ''
df['link'] = ''

# 5. Extract 4-digit year -> published_year
if 'published_date' in df.columns:
    df['published_year'] = (
        df['published_date']
        .fillna('')
        .astype(str)
        .str.extract(r'(\b\d{4}\b)', expand=False)
        .fillna('')
    )
else:
    df['published_year'] = ''

# 6. Clean pages -> Non-negative integer string or empty string ""
if 'pages' in df.columns:
    pages_num = pd.to_numeric(df['pages'], errors='coerce')
    df['pages'] = pages_num.apply(lambda v: str(int(v)) if pd.notna(v) and v >= 0 else '')
else:
    df['pages'] = ''

# 7. Clean ISBN
if 'isbn' in df.columns:
    def format_isbn(val):
        if pd.isna(val):
            return ''
        s = str(val).strip()
        if s.lower() in ['nan', 'none', '<na>', '']:
            return ''
        if '.' in s or 'e' in s.lower():
            try:
                s = str(int(float(s)))
            except (ValueError, OverflowError):
                if s.endswith('.0'):
                    s = s[:-2]
        return s

    df['isbn'] = df['isbn'].apply(format_isbn)
else:
    df['isbn'] = ''

# 8. Select exact Zod Schema fields
schema_fields = [
    'title',
    'author',
    'categories',
    'thumbnail',
    'description',
    'pages',
    'publisher',
    'language',
    'link',
    'published_year',
    'isbn'
]

processed_df = df[schema_fields]

# 9. STRICT FILTERING: Enforce title >= 3, author >= 3, isbn >= 1
valid_mask = (
    (processed_df['title'].str.len() >= 3) &
    (processed_df['author'].str.len() >= 3) &
    (processed_df['isbn'].str.len() >= 1)
)

final_df = processed_df[valid_mask].reset_index(drop=True)

# 10. Save to CSV
os.makedirs(os.path.dirname(output_path), exist_ok=True)
final_df.to_csv(output_path, index=False)

print(f"Total raw records read: {len(df)}")
print(f"Records fully passing Zod schema saved to data4.csv: {len(final_df)}")

Total raw records read: 1299
Records fully passing Zod schema saved to data4.csv: 1287


In [19]:
import os
import pandas as pd
import numpy as np

# File paths
input_path = r"D:\projects\library search engine\backend\process_raw_data\google_books_dataset.csv"
output_path = r"D:\projects\library search engine\backend\process_raw_data\data5.csv"

# 1. Read Raw CSV as strings
df = pd.read_csv(input_path, dtype=str)

# 2. Map original dataset columns to Zod schema keys
col_map = {
    'title': 'title',
    'authors': 'author',
    'categories': 'categories',
    'thumbnail': 'thumbnail',
    'description': 'description',
    'page_count': 'pages',
    'publisher': 'publisher',
    'language': 'language',
    'info_link': 'link'
}
df = df.rename(columns=col_map)

# 3. Clean String Fields & Strip Whitespace
str_cols = ['title', 'author', 'categories', 'thumbnail', 'description', 'publisher', 'language', 'link']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].fillna('').astype(str).str.strip()
        df[col] = df[col].replace({'none': '', 'nan': '', 'None': '', '<NA>': ''})
    else:
        df[col] = ''

# 4. Extract 4-digit year -> published_year
if 'published_date' in df.columns:
    df['published_year'] = (
        df['published_date']
        .fillna('')
        .astype(str)
        .str.extract(r'(\b\d{4}\b)', expand=False)
        .fillna('')
    )
else:
    df['published_year'] = ''

# 5. Clean pages -> Non-negative integer string or empty string ""
if 'pages' in df.columns:
    pages_num = pd.to_numeric(df['pages'], errors='coerce')
    df['pages'] = pages_num.apply(lambda v: str(int(v)) if pd.notna(v) and v >= 0 else '')
else:
    df['pages'] = ''

# 6. Extract ISBN (Prefer isbn_13 -> fallback isbn_10)
def extract_required_isbn(row):
    for col in ['isbn_13', 'isbn_10']:
        if col in row:
            val = str(row.get(col, '')).strip()
            if val and val.lower() not in ['nan', 'none', '<na>', '']:
                if '.' in val or 'e' in val.lower():
                    try:
                        val = str(int(float(val)))
                    except (ValueError, OverflowError):
                        if val.endswith('.0'):
                            val = val[:-2]
                return val
    return ''

df['isbn'] = df.apply(extract_required_isbn, axis=1)

# 7. Select exact Zod Schema fields
schema_fields = [
    'title',
    'author',
    'categories',
    'thumbnail',
    'description',
    'pages',
    'publisher',
    'language',
    'link',
    'published_year',
    'isbn'
]

processed_df = df[schema_fields]

# 8. STRICT FILTERING: Enforce title >= 3, author >= 3, isbn >= 1
valid_mask = (
    (processed_df['title'].str.len() >= 3) &
    (processed_df['author'].str.len() >= 3) &
    (processed_df['isbn'].str.len() >= 1)
)

final_df = processed_df[valid_mask].reset_index(drop=True)

# 9. Save to CSV
os.makedirs(os.path.dirname(output_path), exist_ok=True)
final_df.to_csv(output_path, index=False)

print(f"Total raw records read: {len(df)}")
print(f"Records fully passing Zod schema saved to data5.csv: {len(final_df)}")

Total raw records read: 15147
Records fully passing Zod schema saved to data5.csv: 7238


In [25]:
import os
import pandas as pd

# Directory path
base_dir = r"D:\projects\library search engine\backend\process_raw_data"
output_file_path = os.path.join(base_dir, "final_combined_books.csv")

# List of input file paths
file_paths = [os.path.join(base_dir, f"data{i}.csv") for i in range(1, 6)]

dataframes = []

# Load all CSV files
for path in file_paths:
    if os.path.exists(path):
        # Load as string (dtype=str) to protect ISBNs and formatting
        df = pd.read_csv(path, dtype=str)
        dataframes.append(df)
        print(f"Loaded {os.path.basename(path)}: {len(df)} rows")
    else:
        print(f"Warning: File not found: {path}")

if dataframes:
    # 1. Vertically concatenate
    combined_df = pd.concat(dataframes, ignore_index=True)

    # 2. Shuffle rows (random_state=42 ensures reproducibility)
    shuffled_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

    # 3. Save merged & shuffled output CSV
    shuffled_df.to_csv(output_file_path, index=False)

    print("\n--- Summary ---")
    print(f"Total merged records: {len(shuffled_df)}")
    print(f"Output saved to: {output_file_path}")
else:
    print("No CSV files found to concatenate.")

Loaded data1.csv: 394 rows
Loaded data2.csv: 89887 rows
Loaded data3.csv: 19251 rows
Loaded data4.csv: 1287 rows
Loaded data5.csv: 7238 rows

--- Summary ---
Total merged records: 118057
Output saved to: D:\projects\library search engine\backend\process_raw_data\final_combined_books.csv


In [27]:
import os
import pandas as pd

# File paths
file_path = r"D:\projects\library search engine\backend\process_raw_data\final_combined_books.csv"
temp_output_path = r"D:\projects\library search engine\backend\process_raw_data\final_combined_books_english.csv"

if not os.path.exists(file_path):
    print(f"Error: File not found at {file_path}")
else:
    # 1. Load dataset
    df = pd.read_csv(file_path, dtype=str)
    initial_count = len(df)

    # 2. Select text fields
    check_cols = [col for col in ["title", "author", "description", "categories", "publisher"] if col in df.columns]

    # 3. Combine text columns
    combined_text = df[check_cols].fillna("").astype(str).sum(axis=1)

    # 4. Keep ASCII-only standard characters (\x00-\x7F)
    ascii_mask = ~combined_text.str.contains(r'[^\x00-\x7F]', regex=True)
    filtered_df = df[ascii_mask].reset_index(drop=True)

    # 5. Save to a fresh output file to prevent write lock conflicts
    filtered_df.to_csv(temp_output_path, index=False)

    removed_count = initial_count - len(filtered_df)
    print("--- Filtering Summary ---")
    print(f"Initial rows:          {initial_count}")
    print(f"Cleaned English rows: {len(filtered_df)}")
    print(f"Removed rows:         {removed_count}")
    print(f"Successfully saved to: {temp_output_path}")

--- Filtering Summary ---
Initial rows:          118057
Cleaned English rows: 76037
Removed rows:         42020
Successfully saved to: D:\projects\library search engine\backend\process_raw_data\final_combined_books_english.csv


In [1]:
import pandas as pd
import re

FILE_PATH = r"D:\projects\library search engine\backend\process_raw_data\final_combined_books_english.csv"


def is_valid_isbn10(isbn):
    isbn = isbn.upper().replace("-", "").replace(" ", "")

    if len(isbn) != 10:
        return False

    if not re.fullmatch(r"\d{9}[\dX]", isbn):
        return False

    total = 0

    for i, char in enumerate(isbn):
        value = 10 if char == "X" else int(char)
        total += value * (10 - i)

    return total % 11 == 0


def is_valid_isbn13(isbn):
    isbn = isbn.replace("-", "").replace(" ", "")

    if len(isbn) != 13 or not isbn.isdigit():
        return False

    # ISBN-13 must start with 978 or 979
    if not isbn.startswith(("978", "979")):
        return False

    total = 0

    for i, digit in enumerate(isbn[:12]):
        weight = 1 if i % 2 == 0 else 3
        total += int(digit) * weight

    check_digit = (10 - (total % 10)) % 10

    return check_digit == int(isbn[12])


def isbn10_to_isbn13(isbn):
    isbn = isbn.upper().replace("-", "").replace(" ", "")

    # Remove ISBN-10 check digit
    isbn13_base = "978" + isbn[:9]

    total = 0

    for i, digit in enumerate(isbn13_base):
        weight = 1 if i % 2 == 0 else 3
        total += int(digit) * weight

    check_digit = (10 - (total % 10)) % 10

    return isbn13_base + str(check_digit)


def clean_isbn(value):
    if pd.isna(value):
        return None

    isbn = str(value).strip().upper()

    # Remove common formatting
    isbn = isbn.replace("-", "").replace(" ", "")

    # Empty / obvious invalid values
    if not isbn:
        return None

    # ISBN-13
    if is_valid_isbn13(isbn):
        return isbn

    # ISBN-10
    if is_valid_isbn10(isbn):
        return isbn10_to_isbn13(isbn)

    return None


# Read CSV
df = pd.read_csv(FILE_PATH, dtype=str, keep_default_na=False)

# Clean ISBN
df["isbn"] = df["isbn"].apply(clean_isbn)

# Keep only rows having a valid ISBN
df = df[df["isbn"].notna()].copy()

# Save back to the SAME file
df.to_csv(FILE_PATH, index=False)

print(f"Cleaning completed.")
print(f"Valid books remaining: {len(df)}")
print(f"File updated: {FILE_PATH}")

Cleaning completed.
Valid books remaining: 22890
File updated: D:\projects\library search engine\backend\process_raw_data\final_combined_books_english.csv


In [2]:
import pandas as pd

input_file = r"D:\projects\library search engine\backend\process_raw_data\final_combined_books_english.csv"
output_file = r"D:\projects\library search engine\backend\process_raw_data\final_combined_books_english_subset_500_books.csv"

# Read the CSV
df = pd.read_csv(input_file)

# Shuffle and select 500 rows
subset = df.sample(n=500, random_state=42)

# Save to a new CSV
subset.to_csv(output_file, index=False)

print(f"Original rows: {len(df)}")
print(f"Subset rows: {len(subset)}")
print(f"Saved to: {output_file}")

Original rows: 22890
Subset rows: 500
Saved to: D:\projects\library search engine\backend\process_raw_data\final_combined_books_english_subset_500_books.csv
